# 01 Peta data: kode diacak, kode tetangga BM25, leakage bins
Membuat maps/*.json yang dipakai notebook 02, 03, dan 11 (permutasi kode seed 42, tetangga BM25, kuintil kebocoran). Hanya CPU.

## Setup
Pakai kernel Python dari ~/venvs/dq (venv yang menjalankan semua skrip). Sel ini menyamakan lingkungan dengan bg.sh:
cwd di codeqa_exp/code, pylib_metrics di path, Java untuk METEOR, dan GPU 0.

In [ ]:
import os, sys, json
from pathlib import Path

HOME = Path.home()
EXP = HOME / "inahuy/kacuy/codeqa_exp"
CODE = EXP / "code"
os.chdir(CODE)  # scripts import kc_common / registry from here
for p in (str(CODE), str(EXP / "pylib_metrics")):
    if p not in sys.path:
        sys.path.insert(0, p)
os.environ["PYTHONPATH"] = f"{EXP / 'pylib_metrics'}:{os.environ.get('PYTHONPATH', '')}"
os.environ["PATH"] = f"{HOME / 'envs/kc_tools/bin'}:{os.environ['PATH']}"  # Java for METEOR
os.environ["JAVA_HOME"] = str(HOME / "envs/kc_tools")
os.environ.setdefault("CUDA_VISIBLE_DEVICES", "0")
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
print("cwd:", os.getcwd(), "| python:", sys.executable)

## make_maps.py
Tanpa argumen; menulis ke codeqa_exp/maps.

In [ ]:
ARGS = []
sys.argv = ['make_maps.py']

**Sumber:** code/make_maps.py

```text
Shared mappings and CPU audits: E2 shuffled code, E3 BM25 neighbour code, E6 leakage bins, E8 study IDs.

BM25 (Okapi, k1=1.5, b=0.75, idf = ln(1 + (N - n + 0.5) / (n + 0.5))) over lowercase whitespace code
tokens of the UNIQUE training code strings. Query term frequencies weight the sum (equivalent to
iterating duplicate query tokens). Normalised similarity = score / self-score of the query under
the same training statistics.
```

In [ ]:
from collections import Counter, defaultdict
import random

import numpy as np
from scipy import sparse

import kc_common as kc

In [ ]:
K1, B, SEED = 1.5, 0.75, 42
OUT = kc.EXP / "maps"
OUT.mkdir(parents=True, exist_ok=True)

train, test = kc.load_split("train"), kc.load_split("test")
toks = lambda s: s.lower().split()

In [ ]:
# ---------- E2: derangement with differing donor code strings ----------
n = len(test)
codes = [r["code"] for r in test]
rng = random.Random(SEED)
perm = list(range(n))
for attempt in range(1, 100001):
    rng.shuffle(perm)
    if all(perm[i] != i and codes[perm[i]] != codes[i] for i in range(n)):
        break
else:
    raise RuntimeError("no valid derangement found")
kc.write_json(OUT / "shuffled_mapping.json", {
    "seed": SEED, "n": n, "attempts": attempt,
    "method": "random.Random(42).shuffle repeated until perm[i] != i and code[perm[i]] != code[i] for all i",
    "test_code_sha256": kc.sha256_file(kc.DATA / "test/test.code"),
    "donor_test_id": perm})
print(f"E2 derangement after {attempt} shuffles")

In [ ]:
# ---------- BM25 index over unique training code strings ----------
unique_codes, code_to_train_ids = [], defaultdict(list)
for r in train:
    if r["code"] not in code_to_train_ids:
        unique_codes.append(r["code"])
    code_to_train_ids[r["code"]].append(r["id"])
vocab = {}
doc_counts = [Counter(toks(c)) for c in unique_codes]
for counts in doc_counts:
    for t in counts:
        vocab.setdefault(t, len(vocab))
N, V = len(unique_codes), len(vocab)
df = np.zeros(V)
for counts in doc_counts:
    for t in counts:
        df[vocab[t]] += 1
idf = np.log(1 + (N - df + 0.5) / (df + 0.5))
doc_len = np.array([sum(c.values()) for c in doc_counts], dtype=float)
avgdl = doc_len.mean()
rows_i, cols_i, vals = [], [], []
for d, counts in enumerate(doc_counts):
    norm = K1 * (1 - B + B * doc_len[d] / avgdl)
    for t, tf in counts.items():
        j = vocab[t]
        rows_i.append(d); cols_i.append(j); vals.append(idf[j] * tf * (K1 + 1) / (tf + norm))
D = sparse.csr_matrix((vals, (rows_i, cols_i)), shape=(N, V), dtype=np.float32)

q_counts = [Counter(toks(r["code"])) for r in test]
qr, qc, qv, self_scores = [], [], [], []

In [ ]:
for i, counts in enumerate(q_counts):
    length = sum(counts.values())
    norm = K1 * (1 - B + B * length / avgdl)
    s = 0.0
    for t, tf in counts.items():
        if t in vocab:
            j = vocab[t]
            qr.append(i); qc.append(j); qv.append(tf)
            s += tf * idf[j] * tf * (K1 + 1) / (tf + norm)
    self_scores.append(s)
Q = sparse.csr_matrix((qv, (qr, qc)), shape=(n, V), dtype=np.float32)
token_sets = [set(c) for c in doc_counts]

In [ ]:
def jaccard(a, b):
    return len(a & b) / len(a | b) if a or b else 1.0


def f1(a, b):
    a, b = a.lower().split(), b.lower().split()
    common = sum((Counter(a) & Counter(b)).values())
    return 2 * common / (len(a) + len(b)) if a or b else 1.0

In [ ]:
neighbour, leakage = [None] * n, [None] * n
for start in range(0, n, 500):
    S = (Q[start:start + 500] @ D.T).toarray()
    for k in range(S.shape[0]):
        i = start + k
        scores = S[k]
        order = np.lexsort((np.arange(N), -scores))  # score desc, then index asc (deterministic ties)
        qset = set(q_counts[i])
        # E6: plain top-1, no exclusion
        top = int(order[0])
        same_code = code_to_train_ids[unique_codes[top]]
        best_tid = max(same_code, key=lambda t: (f1(train[t]["question"], test[i]["question"]), -t))
        leakage[i] = {"id": i, "top1_unique_index": top, "top1_train_id": best_tid,
                      "bm25": float(scores[top]), "bm25_norm": float(scores[top] / self_scores[i]) if self_scores[i] else 0.0,
                      "jaccard": jaccard(qset, token_sets[top]), "exact_code_match": unique_codes[top] == test[i]["code"],
                      "question_f1_to_matched": f1(train[best_tid]["question"], test[i]["question"]),
                      "answer_f1_to_matched": f1(train[best_tid]["answer"], test[i]["answer"]),
                      "max_answer_f1_same_code": max(f1(train[t]["answer"], test[i]["answer"]) for t in same_code)}
        # E3: first candidate that is neither identical nor a near duplicate (Jaccard > 0.9)
        for rank, d in enumerate(order):
            d = int(d)
            jac = jaccard(qset, token_sets[d])
            if unique_codes[d] != test[i]["code"] and jac <= 0.9:
                neighbour[i] = {"id": i, "neighbour_unique_index": d, "neighbour_train_id": code_to_train_ids[unique_codes[d]][0],
                                "rank_after_exclusion": rank, "bm25": float(scores[d]),
                                "bm25_norm": float(scores[d] / self_scores[i]) if self_scores[i] else 0.0, "jaccard": jac}
                break
    print(f"BM25 {min(start + 500, n)}/{n}", flush=True)

assert all(x is not None for x in neighbour)
kc.write_json(OUT / "neighbour_mapping.json", {
    "method": "BM25 Okapi k1=1.5 b=0.75 over lowercase whitespace tokens of unique train code strings; "
              "skip identical code and normalized-token Jaccard > 0.9; ties broken by lower index",
    "n_unique_train_codes": N, "vocab_size": V, "avgdl": avgdl, "mapping": neighbour})

norm = np.array([x["bm25_norm"] for x in leakage])

In [ ]:
edges = np.percentile(norm, [20, 40, 60, 80]).tolist()
for x in leakage:
    x["quintile"] = int(np.searchsorted(edges, x["bm25_norm"], side="right")) + 1
kc.write_json(OUT / "leakage_bins.json", {
    "method": "E6: BM25 top-1 training code (no exclusion); matched train example = same code with highest "
              "question token-F1 to the test question; quintiles on bm25_norm",
    "quintile_edges_bm25_norm": edges, "items": leakage})
print("E6 quintile edges:", [round(e, 4) for e in edges])
print("E6 exact code match:", sum(x["exact_code_match"] for x in leakage))

In [ ]:
# ---------- E8: 300 stratified IDs ----------
yn = [r["id"] for r in test if r["answer"].strip().lower() in ("yes", "no")]
other = [r["id"] for r in test if r["answer"].strip().lower() not in ("yes", "no")]
k_yn = round(300 * len(yn) / n)
rng = random.Random(SEED)
ids = sorted(rng.sample(yn, k_yn) + rng.sample(other, 300 - k_yn))
kc.write_json(OUT / "metric_study_ids.json", {"seed": SEED, "n": 300, "yes_no": k_yn, "other": 300 - k_yn,
                                               "test_yes_no_total": len(yn), "ids": ids})
print(f"E8: {k_yn} yes/no + {300 - k_yn} other")